# Direct Arylation: generalisation check (GP-BO vs. PCV vs. BatchSelect, EI and UCB)

Repeats the candidate-level architecture comparison (**GP-BO**, **PCV**,
**BatchSelect**, each with EI and UCB acquisition) on a second, mechanistically
different reaction -- palladium-catalysed direct C-H arylation (Perera et al.
2018) -- to check whether the Buchwald-Hartwig findings are benchmark-specific
or general. 20 seeds x 20 iterations x 6 methods, `qwen3:30b`.

The dataset is fetched directly from its public CSV mirror in the `edbo`
project's GitHub repository (Perera et al. 2018) -- no local file needed.

Writes `results_n20_arylation.pkl`, which feeds Figures 6 and 14 via
`scripts/generate_figures.py`.

**Consolidation note:** the original notebook ran this 20-seed grid in three
incremental batches (seeds 0-4, 5-9, then 10-19), purely so progress could be
checked between working sessions, then concatenated the results. Since each
seed is a fully independent campaign, running all 20 in one loop (as below)
changes nothing about the result -- it only changes how the 20 campaigns are
grouped in wall-clock time. The three intermediate checkpoint files from the
original incremental run (`checkpoint_n5_arylation.pkl`,
`checkpoint_n5_9_arylation.pkl`, `checkpoint_n10_19_arylation.pkl`,
`results_n10_arylation.pkl`) are kept in this directory for provenance but are
not needed to reproduce `results_n20_arylation.pkl`.

Renamed from `dia13_aylation_benchmark.ipynb` (typo fixed).

In [ ]:
from bayesllm.direct_arylation import DirectArylationBenchmark

bench = DirectArylationBenchmark(model="qwen3:30b")
print(f"Feature space dimensionality: {bench.n_dims}")
print(f"Emulator in-sample R2 (sanity check only): {bench.emulator.score(bench.X_scaled, bench.y_raw):.3f}")

## Method-specific orchestration

In [ ]:
def run_multiagent_iteration(X_bo, Y_bo, history, bounds=None, max_rejections=3, iteration=0, seed=None, verbose=False):
    candidate_tensor = bench.propose_candidate(X_bo, Y_bo, bounds, seed=seed)
    return bench.run_pcv_deliberation(candidate_tensor, X_bo, Y_bo, history,
                                       bounds=bounds, max_rejections=max_rejections,
                                       iteration=iteration, verbose=verbose)


def run_multiagent_ucb_iteration(X_bo, Y_bo, history, bounds=None, beta=2.0, max_rejections=3,
                                  iteration=0, seed=None, verbose=False):
    candidate_tensor = bench.propose_candidate_ucb(X_bo, Y_bo, bounds, beta=beta, seed=seed)
    result = bench.run_pcv_deliberation(candidate_tensor, X_bo, Y_bo, history,
                                         bounds=bounds, max_rejections=max_rejections,
                                         iteration=iteration, verbose=verbose)
    result['source'] = 'multiagent_ucb' if result['source'] == 'multiagent' else 'bo_fallback_ucb'
    return result


def run_batch_iteration(X_bo, Y_bo, history, bounds=None, q=4, max_rejections=3, iteration=0, verbose=False):
    candidates_tensor = bench.propose_batch(X_bo, Y_bo, bounds, q=q, seed=iteration)
    candidates_dict_list = [bench.tensor_to_dict(candidates_tensor[i]) for i in range(q)]
    idx, selection_reasoning = bench.call_selector(candidates_dict_list, history, seed=iteration)
    selected_candidate = candidates_tensor[idx].unsqueeze(0)

    result = bench.run_pcv_deliberation(selected_candidate, X_bo, Y_bo, history,
                                         bounds=bounds, max_rejections=max_rejections,
                                         iteration=iteration, verbose=verbose)
    result['source'] = 'batch_llm' if result['source'] == 'multiagent' else 'bo_fallback'
    result['selection_reasoning'] = selection_reasoning
    result['batch_index_selected'] = idx
    return result


def run_batch_llm_ucb_iteration(X_bo, Y_bo, history, bounds=None, q=4, beta=2.0,
                                 max_rejections=3, iteration=0, verbose=False):
    candidates_tensor = bench.propose_batch_ucb(X_bo, Y_bo, bounds, q=q, beta=beta, seed=iteration)
    candidates_dict_list = [bench.tensor_to_dict(candidates_tensor[i]) for i in range(q)]
    idx, selection_reasoning = bench.call_selector(candidates_dict_list, history, seed=iteration)
    selected_candidate = candidates_tensor[idx].unsqueeze(0)

    result = bench.run_pcv_deliberation(selected_candidate, X_bo, Y_bo, history,
                                         bounds=bounds, max_rejections=max_rejections,
                                         iteration=iteration, verbose=verbose)
    result['source'] = 'batch_llm_ucb' if result['source'] == 'multiagent' else 'bo_fallback_ucb'
    result['selection_reasoning'] = selection_reasoning
    result['batch_index_selected'] = idx
    return result


def run_one_iteration(method, X_bo, Y_bo, history, iteration, seed_base):
    iter_seed = seed_base * 1000 + iteration

    if method == "bo_only":
        return bench.run_bo_only_iteration(X_bo, Y_bo, bounds=bench.bounds, seed=iter_seed)
    elif method == "bo_only_ucb":
        return bench.run_bo_only_ucb_iteration(X_bo, Y_bo, bounds=bench.bounds, beta=2.0, seed=iter_seed)
    elif method == "multiagent":
        return run_multiagent_iteration(X_bo, Y_bo, history, bounds=bench.bounds, max_rejections=3,
                                         iteration=iteration, seed=iter_seed, verbose=False)
    elif method == "multiagent_ucb":
        return run_multiagent_ucb_iteration(X_bo, Y_bo, history, bounds=bench.bounds, beta=2.0,
                                             max_rejections=3, iteration=iteration, seed=iter_seed, verbose=False)
    elif method == "batch_llm":
        return run_batch_iteration(X_bo, Y_bo, history, bounds=bench.bounds, q=4,
                                    max_rejections=3, iteration=iteration, verbose=False)
    elif method == "batch_llm_ucb":
        return run_batch_llm_ucb_iteration(X_bo, Y_bo, history, bounds=bench.bounds, beta=2.0,
                                            max_rejections=3, iteration=iteration, verbose=False)
    else:
        raise ValueError(f"Unknown method: {method}")

## Seeds loop: experimental configuration and main run

In [ ]:
import time
import pickle
import torch
import pandas as pd

METHODS_ARYLATION = ["bo_only", "bo_only_ucb", "multiagent", "multiagent_ucb", "batch_llm", "batch_llm_ucb"]
SEEDS_ARYLATION = list(range(20))
N_ITER_ARYLATION = 20
CHECKPOINT_PATH_ARYLATION = "results_n20_arylation.pkl"

results_n20_arylation = []

for seed in SEEDS_ARYLATION:
    X_init, Y_init, history_init = bench.generate_initial_design(seed)
    for method in METHODS_ARYLATION:
        X_bo, Y_bo, history = bench.clone_run_state(X_init, Y_init, history_init)
        t0 = time.time()
        for iteration in range(N_ITER_ARYLATION):
            result = run_one_iteration(method, X_bo, Y_bo, history, iteration, seed_base=seed)
            x_new, y_new = result['x'], result['y']
            X_bo = torch.cat([X_bo, x_new])
            Y_bo = torch.cat([Y_bo, y_new])
            history.append({'descriptors': bench.tensor_to_dict(x_new), 'yield': y_new.item()})
            results_n20_arylation.append({
                'method': method, 'seed': seed, 'iteration': iteration,
                'yield': y_new.item(), 'source': result['source'], 'rejections': result['rejections'],
            })
            with open(CHECKPOINT_PATH_ARYLATION, "wb") as f:
                pickle.dump(results_n20_arylation, f)
        print(f"[seed={seed}][{method}] done in {(time.time()-t0)/60:.1f} min")

df_check = pd.DataFrame(results_n20_arylation)
print("\nTotal rows:", len(df_check))
print("Unique (method, seed, iteration):", df_check[["method", "seed", "iteration"]].drop_duplicates().shape[0])
print("Rows per method:\n", df_check.groupby("method").size())

## Post-hoc analysis: paired Wilcoxon vs. GP-BO

In [ ]:
from scipy.stats import wilcoxon

df_arylation_results = pd.DataFrame(results_n20_arylation)
df_arylation_results = df_arylation_results.sort_values(["method", "seed", "iteration"])
df_arylation_results["best_so_far"] = df_arylation_results.groupby(["method", "seed"])["yield"].cummax()

final_iter = df_arylation_results["iteration"].max()
final_best_arylation = df_arylation_results[df_arylation_results["iteration"] == final_iter][["method", "seed", "best_so_far"]]
pivot_arylation = final_best_arylation.pivot(index="seed", columns="method", values="best_so_far")
print(pivot_arylation.round(2))
print("\nMean final best-so-far by method:")
print(pivot_arylation.mean().sort_values(ascending=False).round(2))

def paired_wilcoxon_arylation(pivot_df, method_a, method_b):
    a, b = pivot_df[method_a].values, pivot_df[method_b].values
    stat, p = wilcoxon(a, b)
    return {"comparison": f"{method_a} vs {method_b}", "mean_a": a.mean(), "mean_b": b.mean(),
            "mean_diff": (a - b).mean(), "n_favoring_a": int(((a - b) > 0).sum()),
            "n_favoring_b": int(((a - b) < 0).sum()), "wilcoxon_p": p}

comparisons_arylation = [
    ("bo_only", "multiagent"), ("bo_only_ucb", "multiagent_ucb"),
    ("bo_only", "batch_llm"), ("bo_only_ucb", "batch_llm_ucb"),
]
print(pd.DataFrame([paired_wilcoxon_arylation(pivot_arylation, a, b) for a, b in comparisons_arylation]).round(4))